# Lab — Early Warning per la Retention della Clientela

**Corso Python per la Data Science · Modulo conclusivo: ML predittivo + LLM via API**

In questo laboratorio costruisci, passo dopo passo, un sistema di *early warning* che
individua i clienti a rischio di abbandono e ne traduce il rischio in una spiegazione e
un'azione per il gestore. Lavori sul dataset `clienti_banca_churn.csv`.

---

### Come si usa questo notebook

Le celle sono di due tipi:

- 💡 **celle già pronte**: leggile, eseguile, *capisci cosa stampano*. Spesso mostrano la
  stessa operazione in due modi: prima **as-is** (esplicito, si vede ogni passaggio) e poi
  **idiomatico** (la versione concisa che useresti davvero). Guarda sempre l'as-is prima.
- ✏️ **`# TODO` — tocca a te**: completa la riga mancante. Sono brevi.

E le domande:

- 🎯 **domande di decisione**: la risposta nasce **dal TUO output** (il numero che esce dalla
  *tua* esecuzione) e **dal TUO giudizio di business**. Non c'è una risposta nel testo da
  copiare: ne discuteremo a voce. Un assistente AI non ha questi numeri e non conosce il
  *vostro* contesto, quindi qui non vi sostituisce.

> **Nota sulla riproducibilità:** fissiamo `random_state=7` ovunque, così tutti ottengono
> gli stessi numeri e possiamo confrontarci.

In [ ]:
# 💡 Librerie e impostazioni di base
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

SEED = 7                       # fissato per avere risultati riproducibili in aula
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
print("Ambiente pronto.")

In [ ]:
# 💡 Carichiamo i dati e diamo una prima occhiata
df = pd.read_csv("clienti_banca_churn.csv")

print("Righe (clienti):", df.shape[0])
print("Colonne:", df.shape[1])
df.head()

### 🎯 Domanda 1 (riscaldamento)

Guarda l'output qui sopra: **quanti clienti** ci sono nel dataset e **quante colonne**?
Apri `df.head()` e scorri le colonne: quali ti sembrano già, a intuito, possibili "spie"
di un cliente che sta per andarsene? (Annotale mentalmente, le verifichi più avanti.)

## 1 · Esplorazione: chi abbandona è diverso?

Prima ancora del modello, vediamo se chi abbandona ha comportamenti diversi da chi resta.
La colonna obiettivo è **`abbandono`** (1 = ha lasciato la banca, 0 = è rimasto).

Cominciamo dal *churn rate per segmento*. Lo facciamo prima **as-is** (esplicito) e poi
nella versione **idiomatica** pandas: stesso risultato, una riga.

In [ ]:
# 💡 AS-IS — calcolo esplicito del tasso di abbandono per segmento
#    Scorriamo i segmenti uno per uno e calcoliamo a mano la proporzione di abbandoni.
segmenti = df["segmento"].unique()
for s in segmenti:
    sottogruppo = df[df["segmento"] == s]          # solo i clienti di quel segmento
    n_totali = len(sottogruppo)
    n_abbandoni = sottogruppo["abbandono"].sum()    # quanti hanno abbandonato
    tasso = n_abbandoni / n_totali
    print(f"{s:10s}  clienti={n_totali:5d}  churn={tasso:.1%}")

In [ ]:
# 💡 IDIOMATICO — stessa cosa con groupby: una riga, stesso risultato
#    'abbandono' e' 0/1, quindi la sua MEDIA per gruppo E' il tasso di abbandono.
df.groupby("segmento")["abbandono"].mean().sort_values(ascending=False)

💡 Le due celle danno lo stesso risultato: l'as-is rende visibile *cosa* stai
calcolando, l'idiomatico è ciò che scriverai una volta capito il meccanismo.

### 🎯 Domanda 2

Quale segmento ha il **tasso di abbandono più alto**? È quello che ti aspettavi?
Pensa a cosa significa per la banca trattenere un cliente di quel segmento rispetto a un altro
(suggerimento: non tutti i clienti "valgono" uguale in termini di patrimonio e ricavi).

In [ ]:
# ✏️ TODO — tocca a te
#    Calcola la MEDIA di 'transazioni_mese' per chi resta (abbandono=0)
#    e per chi abbandona (abbandono=1).
#    Suggerimento: e' lo stesso schema della cella idiomatica qui sopra,
#    ma raggruppando per 'abbandono' invece che per 'segmento'.

# df.groupby( ... )[ ... ].mean()


### 🎯 Domanda 3

In base al tuo risultato: chi abbandona fa **più** o **meno** transazioni al mese di chi resta?
Prova a immaginare *prima* di costruire il modello: quali variabili pensi che il modello userà
di più per riconoscere un cliente a rischio?

## 2 · Preparazione dei dati

Un modello di scikit-learn vuole **solo numeri** e **niente valori mancanti**. Quindi:

1. la colonna `nps` ha alcuni valori mancanti → li riempiamo con la mediana;
2. le colonne testuali categoriche (`segmento`, `canale_preferito`) → le trasformiamo in
   colonne 0/1 con il *one-hot encoding* (`pd.get_dummies`);
3. togliamo ciò che non è una feature predittiva: `id_cliente` (è solo un'etichetta),
   `abbandono` (è la risposta, non un indizio) e `note_gestore` (testo libero, lo useremo
   dopo con l'LLM).

In [ ]:
# 💡/✏️ Preparazione. Una riga e' da completare (vedi TODO).
df["nps"] = df["nps"].fillna(df["nps"].median())   # 1) riempiamo i mancanti di nps

# 2) separiamo le feature (X) dalla risposta (y)
X = df.drop(columns=["id_cliente", "abbandono", "note_gestore"])
y = df["abbandono"]

# ✏️ TODO — completa il one-hot encoding delle due colonne categoriche.
#    Suggerimento: pd.get_dummies(X, columns=[...], drop_first=True)
# X = pd.get_dummies(X, columns=[ ... ], drop_first=True)

print("Feature dopo la codifica:", X.shape[1])
X.head(3)

In [ ]:
# 💡 Train/test split STRATIFICATO.
#    'stratify=y' mantiene la stessa proporzione di abbandoni nei due insiemi:
#    fondamentale quando una classe e' rara (qui gli abbandoni sono ~14%).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
print("Train:", len(y_train), " Test:", len(y_test))
print("Abbandoni reali nel test:", int(y_test.sum()), "su", len(y_test))

## 3 · Il modello (il "cervello")

Usiamo un **Random Forest**: robusto, gestisce bene variabili eterogenee e ci dà una misura
di *importanza* delle feature. Mettiamo `class_weight="balanced"` perché gli abbandoni sono
pochi e non vogliamo che il modello impari semplicemente a dire "non abbandona" per tutti.

In [ ]:
# ✏️ TODO — addestra il modello.
#    Il modello e' gia' configurato: ti manca solo la chiamata .fit(...) sui dati di TRAIN.
modello = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=SEED
)

# modello.fit( ... , ... )      # <-- completa con X_train e y_train

print("Modello addestrato.")

In [ ]:
# 💡 Probabilita' di abbandono per ogni cliente del test, e qualita' complessiva
prob = modello.predict_proba(X_test)[:, 1]    # P(abbandono) tra 0 e 1
auc = roc_auc_score(y_test, prob)
print(f"ROC-AUC sul test: {auc:.3f}   (0.5 = caso, 1.0 = perfetto)")

### 🎯 Domanda 4

Qual è il **ROC-AUC** che hai ottenuto? Un valore nettamente sopra **0,5** significa che il
modello ha trovato segnale vero nei dati (non sta tirando a indovinare). Secondo te basta un
buon ROC-AUC per mettere il sistema in produzione, o manca ancora qualcosa? (Tieni il pensiero:
la prossima sezione è la risposta.)

## 4 · Il punto di decisione: la soglia ⭐

Il modello non dice "abbandona / non abbandona": dice una **probabilità** (es. 0,23).
Per agire dobbiamo scegliere una **soglia**: *sopra quale probabilità chiamiamo il cliente?*

Questa **non è una scelta statistica, è una scelta di business**: una soglia alta intercetta
pochi clienti ma quasi tutti veri a rischio; una soglia bassa ne intercetta molti di più, ma
con più "falsi allarmi" (clienti che non sarebbero andati via). Vediamolo coi numeri.

In [ ]:
# 💡 Funzione che valuta una soglia: quanti abbandoni intercetta e quanti falsi allarmi
def valuta_soglia(soglia):
    pred = (prob >= soglia).astype(int)          # 1 se prob >= soglia, altrimenti 0
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    intercettati = tp                            # abbandoni veri che abbiamo "preso"
    persi = fn                                   # abbandoni veri che ci sono sfuggiti
    falsi_allarmi = fp                           # clienti tranquilli segnalati per errore
    recall = tp / (tp + fn)                      # quota di abbandoni intercettati
    print(f"Soglia {soglia:.2f}  ->  intercettati {intercettati}/{tp+fn} abbandoni "
          f"(recall {recall:.0%}) | falsi allarmi: {falsi_allarmi}")
    return intercettati, persi, falsi_allarmi

valuta_soglia(0.50)

### 🎯 Domanda 5

A soglia **0,50** (la scelta "di default", apparentemente neutra): **quanti** abbandoni reali
intercetta il modello, su tutti quelli presenti nel test? Ti sembra un sistema utile per il
business, così com'è?

In [ ]:
# ✏️ TODO — prova soglie piu' basse e osserva come cambiano i numeri.
#    Chiama valuta_soglia(...) con 0.30 e poi con 0.20.

# valuta_soglia( ... )
# valuta_soglia( ... )


### 🎯 Decisione (il cuore del lab)

Hai visto tre scenari. Ora **scegli tu** la soglia, ragionando da analista di business.

Ipotesi di lavoro (puoi adattarle):

- contattare un cliente per un'azione di retention costa circa **50 € a cliente** (tempo del
  gestore, eventuale offerta);
- trattenere un cliente *affluent* vale molto di più che trattenerne uno *mass* (patrimonio in
  gestione + ricavi commissionali ricorrenti).

Domande a cui rispondere **nella cella sotto**:

1. Quale soglia scegli e **perché**? Cosa stai ottimizzando (intercettare più abbandoni? limitare
   i falsi allarmi e quindi il costo?)
2. Useresti la **stessa soglia per tutti i segmenti**, o una più aggressiva per gli *affluent*
   e i *private*? Motiva.
3. Cosa diresti al responsabile commerciale che ti chiede "ma perché non mettiamo 0,50 che
   prende solo quelli sicuri"?

> Non c'è una risposta "giusta" universale: c'è una risposta *difendibile*. Preparati a
> sostenerla a voce in 1 minuto.

**La tua decisione** *(scrivi qui sotto facendo doppio clic)*

- Soglia scelta: ____
- Perché / cosa ottimizzo: ____
- Stessa soglia per tutti i segmenti? ____
- Risposta al "perché non 0,50": ____

## 5 · Interpretabilità: quali sono i driver del rischio?

Il modello non solo predice: ci dice **quali feature pesano di più**. Questo è ciò che, più
avanti, daremo in pasto all'LLM perché lo traduca in parole. (Attenzione: la *feature
importance* è globale, vale sul modello in generale; per spiegare il *singolo* cliente si usano
tecniche come SHAP — è uno degli spunti di approfondimento.)

In [ ]:
# ✏️ TODO — ordina le importanze in modo decrescente.
#    modello.feature_importances_ e' un array allineato a X.columns.
importanze = pd.Series(modello.feature_importances_, index=X.columns)

# importanze = importanze.sort_values( ... )      # <-- decrescente

print(importanze.head(8))

# Grafico (gia' pronto): le prime 10 feature per importanza
importanze.head(10).iloc[::-1].plot(kind="barh")
plt.title("Driver del rischio di abbandono (feature importance)")
plt.xlabel("importanza")
plt.tight_layout()
plt.show()

### 🎯 Domanda 6

Guarda il **tuo** grafico: quali sono i **3 driver principali**? Tornano con quello che avevi
intuito nell'esplorazione (Domande 2 e 3)? E soprattutto: hanno **senso di business**? Un driver
che pesa molto ma non ha spiegazione plausibile è un campanello d'allarme (possibile *data leak*
o correlazione spuria).

## 6 · L'LLM: la "voce" del modello 🗣️

Finora il **modello è il cervello**: stima il rischio e i driver. Ora usiamo un **LLM come voce**:
traduce quel segnale in una spiegazione e in un'azione per il gestore.

> ⚠️ **Punto di onestà tecnica.** L'LLM **non** sta spiegando *come ragiona il modello*. Noi gli
> passiamo il rischio e i driver (che vengono dal modello e dall'interpretabilità) e lui li
> **mette in parole**. È un traduttore, non un oracolo.

**Chiave API.** Le celle seguenti funzionano in due modi:
- se hai impostato la variabile d'ambiente `OPENAI_API_KEY`, fanno una **vera** chiamata a
  `gpt-4o-mini`;
- altrimenti usano una **risposta simulata** (così il notebook gira lo stesso in aula). La
  risposta simulata è pensata *apposta* per l'esercizio di critica più sotto.

In [ ]:
# 💡 Configurazione LLM (con fallback simulato se manca la chiave)
import os
api_key = os.environ.get("OPENAI_API_KEY")
try:
    from google.colab import userdata          # in Colab: chiave dai Secret
    api_key = api_key or userdata.get("OPENAI_API_KEY")
except Exception:
    pass                                         # fuori da Colab, o Secret assente

USA_LLM_REALE = bool(api_key)
print("Modalità LLM:", "REALE (gpt-4o-mini)" if USA_LLM_REALE else "SIMULATA (offline)")

if USA_LLM_REALE:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

In [ ]:
# 💡 Scegliamo il cliente a più alto rischio dal test set.
#    Una Series delle probabilità indicizzata come df, poi idxmax().
rischio = pd.Series(prob, index=X_test.index)
indice_top = rischio.idxmax()

cliente_top = df.loc[indice_top]
print("Cliente più a rischio, indice:", indice_top)
print(f"Probabilità di abbandono stimata: {rischio.loc[indice_top]:.0%}")

In [ ]:
# 💡 Prepariamo il "contesto" da passare all'LLM: i dati del cliente + i driver.
#    I driver sono le feature più importanti per il modello, ma SOLO quelle in cui
#    questo cliente è davvero sul lato "a rischio": la direzione del rischio la
#    apprendiamo dai dati (segno della correlazione col target). Così non passiamo
#    all'LLM, come "fattori di rischio", caratteristiche che in realtà lo abbassano.
mediane   = df.median(numeric_only=True)
direzione = df.corr(numeric_only=True)["abbandono"]   # > 0: valore alto = più rischio

driver = []
for f in importanze.sort_values(ascending=False).index:
    if f not in df.columns or f not in direzione.index:
        continue                              # salta le colonne create dal get_dummies
    valore = df.loc[indice_top, f]
    alto_e_rischioso = direzione[f] > 0
    sul_lato_rischioso = valore > mediane[f] if alto_e_rischioso else valore < mediane[f]
    if sul_lato_rischioso:
        driver.append({"feature": f, "valore": round(float(valore), 1)})
    if len(driver) == 3:
        break

cliente = {
    "id_cliente": cliente_top["id_cliente"],
    "segmento": cliente_top["segmento"],
}
print("Cliente:", cliente)
print("Driver passati all'LLM:")
for d in driver:
    print("  -", d["feature"], "=", d["valore"])

In [ ]:
# 💡 RUOLO 1 dell'LLM: spiegare il rischio al gestore (in linguaggio naturale)
def _mock_spiegazione(cliente, prob, driver):
    # Risposta SIMULATA. Contiene DI PROPOSITO un dettaglio non presente nei dati,
    # per l'esercizio di critica della prossima cella.
    return (
        f"Il cliente {cliente['id_cliente']} (segmento {cliente['segmento']}) presenta un "
        f"rischio di abbandono del {prob:.0%}. L'attivita' transazionale risulta in calo e la "
        "soddisfazione e' sotto la media. Si segnala inoltre che il cliente ha contattato il "
        "call center tre volte nell'ultimo mese, a conferma di un crescente malcontento."
    )

def spiega_rischio(cliente, prob, driver):
    """Trasforma l'output del modello in una spiegazione per il gestore."""
    if not USA_LLM_REALE:
        return _mock_spiegazione(cliente, prob, driver)
    fattori = "\n".join(f"- {d['feature']}: {d['valore']}" for d in driver)
    prompt = (
        "Sei un assistente per i gestori di una banca.\n"
        f"Cliente {cliente['id_cliente']} - probabilità di abbandono: {prob:.0%}.\n"
        f"Principali fattori che spingono il rischio:\n{fattori}\n\n"
        "Scrivi 3-4 frasi in italiano che spieghino al gestore perché il cliente è a "
        "rischio. Tono professionale, niente gergo tecnico. Non inventare dati non elencati."
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return resp.choices[0].message.content

In [ ]:
# 💡 Eseguiamo il Ruolo 1
spiegazione = spiega_rischio(cliente, rischio.loc[indice_top], driver)
print(spiegazione)

### 🎯 Task di critica (anti-allucinazioni)

Rileggi la spiegazione qui sopra e **confrontala con i dati che hai effettivamente passato**
all'LLM (li trovi stampati due celle fa, nella lista "Driver").

- C'è qualche affermazione che **non puoi verificare** dai dati forniti? L'LLM ha aggiunto
  dettagli plausibili ma **non presenti** negli input?
- Perché in banca un dettaglio inventato in un messaggio al gestore può essere un problema serio?

> Se stai usando l'LLM **reale** e non trovi invenzioni, prova a **rilanciare alzando
> `temperature`** o togliendo l'istruzione "non inventare": vedrai quanto facilmente un LLM
> "riempie i vuoti". È esattamente il rischio da presidiare (→ *human-in-the-loop*: la
> spiegazione è un suggerimento, la decisione resta al gestore).

In [ ]:
# ✏️ RUOLO 2 dell'LLM: proporre un'azione di retention in formato JSON STRUTTURATO
#    (cosi' e' integrabile in un CRM, non solo leggibile a schermo).
def _mock_azione(cliente, prob, motivo):
    return {"canale": "telefono", "priorita": "alta",
            "messaggio": f"Contattare il cliente {cliente['id_cliente']} per un check-up della relazione."}

def proponi_azione(cliente, prob, motivo):
    """Genera un'azione di retention come oggetto strutturato (JSON)."""
    if not USA_LLM_REALE:
        return _mock_azione(cliente, prob, motivo)
    # ✏️ TODO — completa il prompt: deve chiedere UNA azione concreta e SOLO un JSON
    #    con le chiavi: canale, messaggio, priorita ("alta"|"media"|"bassa").
    prompt = (
        f"Cliente {cliente['id_cliente']}, segmento {cliente['segmento']}, "
        f"rischio di abbandono {prob:.0%}. Motivo principale: {motivo}.\n\n"
        # ... scrivi qui le istruzioni e il formato JSON richiesto ...
        ""
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},   # output garantito in JSON
        temperature=0.4,
    )
    return json.loads(resp.choices[0].message.content)

motivo = f"{driver[0]['feature']} = {driver[0]['valore']}" if driver else "attivita' in calo"
azione = proponi_azione(cliente, rischio.loc[indice_top], motivo)
print(json.dumps(azione, indent=2, ensure_ascii=False))

### 🎯 Domanda 7

L'azione proposta (canale, messaggio, priorità) è **coerente** con il profilo del cliente e col
suo segmento? La passeresti al gestore **così com'è**, o la modificheresti? Perché ti conviene
farti restituire un **JSON** invece di un testo libero?

In [ ]:
# 💡 RUOLO 3 (estensione): l'LLM come estrattore di feature dal testo libero.
#    Molti clienti hanno una 'note_gestore'. La trasformiamo in dati strutturati.
def _mock_estrai(nota):
    n = nota.lower()
    menziona = int(any(k in n for k in ["concorrente", "altro istituto", "altra banca", "trasferire"]))
    sentiment = "negativo" if any(k in n for k in ["lament", "insoddisf", "chiud", "trasferire"]) else "neutro"
    tema = "investimenti" if any(k in n for k in ["dossier", "titoli", "investiment"]) else "altro"
    return {"sentiment": sentiment, "menziona_concorrente": menziona, "tema": tema}

def estrai_segnali(nota):
    """Trasforma testo libero in feature strutturate."""
    if not isinstance(nota, str) or not nota.strip():
        return {"sentiment": "n/d", "menziona_concorrente": 0, "tema": "n/d"}
    if not USA_LLM_REALE:
        return _mock_estrai(nota)
    prompt = (
        "Analizza la nota del gestore e rispondi SOLO con JSON:\n"
        '{"sentiment": "positivo|neutro|negativo", '
        '"menziona_concorrente": 0 o 1, '
        '"tema": "costi|servizio|investimenti|nessuno|altro"}\n\n'
        f'Nota: "{nota}"'
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

# Prendiamo un cliente che ha davvero una nota e proviamo
con_nota = df[df["note_gestore"].notna() & (df["note_gestore"].str.strip() != "")].iloc[0]
print("Nota:", con_nota["note_gestore"])
print("Estratto:", estrai_segnali(con_nota["note_gestore"]))

### 🎯 Domanda 8 (riflessione finale)

Hai visto l'LLM in tre ruoli: **spiegare**, **proporre un'azione**, **estrarre dati dal testo**.
Per ognuno: l'LLM aggiunge valore reale, o un approccio più semplice (una regola, un modello)
basterebbe? In quali dei tre ruoli il rischio di **allucinazione** ti preoccupa di più, e come lo
terresti sotto controllo?

## In sintesi — cosa hai costruito

Hai percorso l'intera catena di un sistema di early warning:

**dati → modello (il cervello) → interpretabilità → LLM (la voce) → azione per il gestore**

con un passaggio aggiuntivo: l'LLM che trasforma le **note testuali** in nuove feature.

### Checklist di consegna
- [ ] Notebook eseguito senza errori (tutti i `# TODO` completati)
- [ ] **Domanda 5**: numero di abbandoni intercettati a soglia 0,50
- [ ] **Decisione soglia** (cella di testo): soglia scelta + motivazione difendibile
- [ ] **Domanda 6**: i 3 driver del rischio nel tuo modello + senso di business
- [ ] **Task di critica**: l'invenzione individuata nell'output dell'LLM
- [ ] Pitch di 1 minuto pronto: "ecco i clienti a rischio, ecco perché, ecco cosa farei"

> Le risposte alle 🎯 nascono dai *tuoi* output e dal *tuo* giudizio. Le confronteremo a voce:
> è lì che si vede se il sistema l'hai capito davvero.